IMPORTING REQUIRED LIBRARIES

In [1]:
import requests
import pandas as pd

USING COINGECKO PUBLIC API


In [2]:
def fetch_crypto_data_coingecko():
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": 50,
        "page": 1,
        "sparkline": "false"
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()
        crypto_list = []

        for coin in data:
            crypto_list.append([
                coin["name"],
                coin["symbol"].upper(),
                coin["current_price"],
                coin["market_cap"],
                coin["total_volume"],
                coin["price_change_percentage_24h"]
            ])

        df = pd.DataFrame(crypto_list, columns=[
            "Cryptocurrency Name", "Symbol", "Current Price (USD)",
            "Market Capitalization", "24h Trading Volume", "24h Price Change (%)"
        ])

        return df
    else:
        print("Failed to fetch data. Status code:", response.status_code)
        return None

USING BINANCE PUBLIC API

In [3]:
def fetch_crypto_data_binance():
    url = "https://api.binance.com/api/v3/ticker/24hr"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        crypto_list = []

        for coin in data[:50]:  # Take top 50 from Binance API
            crypto_list.append([
                coin["symbol"],
                coin["lastPrice"],
                coin["quoteVolume"],
                coin["priceChangePercent"]
            ])

        df = pd.DataFrame(crypto_list, columns=[
            "Symbol", "Current Price (USD)", "24h Trading Volume", "24h Price Change (%)"])

        return df
    else:
        print("Failed to fetch data from Binance. Status code:", response.status_code)
        return None

MERGING BOTH DATAFRAMES

In [4]:
def fetch_crypto_data():
    df_coingecko = fetch_crypto_data_coingecko()
    df_binance = fetch_crypto_data_binance()

    if df_coingecko is not None and df_binance is not None:
        df_merged = pd.merge(df_coingecko, df_binance, on="Symbol", how="outer", suffixes=("_coingecko", "_binance"))
    elif df_coingecko is not None:
        df_merged = df_coingecko
    elif df_binance is not None:
        df_merged = df_binance
    else:
        df_merged = None

    return df_merged



In [5]:
# FINAL RESULT
df_merged = fetch_crypto_data()
if df_merged is not None:
    print("Final Data:")
    print(df_merged.head(10))


Failed to fetch data from Binance. Status code: 451
Final Data:
  Cryptocurrency Name Symbol  Current Price (USD)  Market Capitalization  \
0             Bitcoin    BTC         96212.000000          1904134221010   
1            Ethereum    ETH          2629.920000           316466318325   
2              Tether   USDT             0.999957           141945976697   
3                 XRP    XRP             2.420000           139720808072   
4              Solana    SOL           197.490000            96163338185   
5                 BNB    BNB           647.070000            94285061775   
6                USDC   USDC             0.999900            56066492423   
7            Dogecoin   DOGE             0.256399            37832194101   
8             Cardano    ADA             0.788459            28258461278   
9   Lido Staked Ether  STETH          2627.050000            24788660218   

   24h Trading Volume  24h Price Change (%)  
0         37244394794              -1.81606  
1      

In [6]:
print(df_merged.info())
print(df_merged.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Cryptocurrency Name    50 non-null     object 
 1   Symbol                 50 non-null     object 
 2   Current Price (USD)    50 non-null     float64
 3   Market Capitalization  50 non-null     int64  
 4   24h Trading Volume     50 non-null     int64  
 5   24h Price Change (%)   50 non-null     float64
dtypes: float64(2), int64(2), object(2)
memory usage: 2.5+ KB
None
       Current Price (USD)  Market Capitalization  24h Trading Volume  \
count            50.000000           5.000000e+01        5.000000e+01   
mean           4169.861964           6.201034e+10        2.592230e+09   
std           18979.262792           2.710394e+11        8.526960e+09   
min               0.000010           2.651178e+09        4.001690e+05   
25%               0.999968           3.707642e+09 

SAVING CURRENT DATA INTO AN EXCEL FILE

In [8]:
df_merged.to_excel("cryptocurrency_data.xlsx")

2nd TASK

Data Analysis

In [9]:
# Identifying the top 5 cryptocurrencies by market cap
top_5_market_cap = df_merged.nlargest(5, "Market Capitalization")
print("Top 5 Cryptocurrencies by Market Capitalization:")
print(top_5_market_cap[["Cryptocurrency Name", "Market Capitalization"]])

Top 5 Cryptocurrencies by Market Capitalization:
  Cryptocurrency Name  Market Capitalization
0             Bitcoin          1904134221010
1            Ethereum           316466318325
2              Tether           141945976697
3                 XRP           139720808072
4              Solana            96163338185


In [10]:
# Calculating the average price of the top 50 cryptocurrencies
avg_price = df_merged["Current Price (USD)"].mean()
print("\nAverage Price of Top 50 Cryptocurrencies: $", round(avg_price, 2))


Average Price of Top 50 Cryptocurrencies: $ 4169.86


In [11]:
# Analyzing the highest 24-hour percentage price change
highest_24h_change = df_merged.loc[df_merged["24h Price Change (%)"].idxmax()]
print("\nCryptocurrency with Highest 24h Price Change:")
print(highest_24h_change[["Cryptocurrency Name", "24h Price Change (%)"]])


Cryptocurrency with Highest 24h Price Change:
Cryptocurrency Name         OKB
24h Price Change (%)    0.98814
Name: 46, dtype: object


In [12]:
# Analyzing the lowest 24-hour percentage price change
lowest_24h_change = df_merged.loc[df_merged["24h Price Change (%)"].idxmin()]
print("\nCryptocurrency with Lowest 24h Price Change:")
print(lowest_24h_change[["Cryptocurrency Name", "24h Price Change (%)"]])


Cryptocurrency with Lowest 24h Price Change:
Cryptocurrency Name     Litecoin
24h Price Change (%)    -8.44187
Name: 20, dtype: object
